# Libraries

In [1]:
from graph_creation import create_mesh_dataset, save_database, create_dataset_folders

In [2]:
create_dataset_folders(dataset_folder='datasets')

# Create and save pytorch geometric dataset

In [3]:
# Configure your new catchment
catchment_raw_folder = './raw_datasets_ahr' 
catchment_name = 'ahr_river' 
with_multiscale = True

# Split configuration
start_sim_id = 1
n_train = 1
n_test = 1

# dataset_folder, dataset_name, dataset_dir, with_multiscale, start_sim_id, n_sim
simulation_ids = [
    # ['./raw_datasets_mesh', 'mesh_dataset2','datasets/train', False, 1, 80],
    # ['./raw_datasets_mesh', 'mesh_dataset2','datasets/test', False, 81, 20],
    # ['./raw_datasets_mesh', 'multiscale_mesh_dataset2','datasets/train', True, 1, 80],
    # ['./raw_datasets_mesh', 'multiscale_mesh_dataset2','datasets/test', True, 81, 20],
    # ['./raw_datasets_dk15', 'dijkring_15_big','datasets/test', True, 101, 11],
    # ['./raw_datasets_dk15', 'dijkring_15_big_fine','datasets/test', False, 101, 11],
    [catchment_raw_folder, catchment_name, 'datasets/train', with_multiscale, start_sim_id, n_train],
    [catchment_raw_folder, catchment_name, 'datasets/test', with_multiscale, start_sim_id + n_train, n_test],
]

In [9]:
# Original D-HYDRO workflow kept for reference (commented out as requested).
# for dataset_folder, dataset_name, dataset_dir, with_multiscale, start_sim_id, n_sim in simulation_ids:
#    mesh_dataset = create_mesh_dataset(
#       dataset_folder,
#       n_sim,
#       start_sim_id,
#       with_multiscale=with_multiscale,
#       number_of_multiscales=4,
#    )
#    save_database(mesh_dataset, name=dataset_name, out_path=dataset_dir)

# SFINCS water-level-only workflow (zs/zb -> WD, VX=0, VY=0).
import os
import json
import copy
import pickle
import numpy as np
import torch
import xarray as xr
from scipy.interpolate import griddata
from scipy.optimize import linear_sum_assignment


def _load_template_data(template_pkl_path='./datasets/train/template_marg.pkl'):
    with open(template_pkl_path, 'rb') as f:
        dataset = pickle.load(f)
    if not isinstance(dataset, list) or len(dataset) == 0:
        raise RuntimeError('Template .pkl must contain a non-empty list of Data objects.')
    return dataset[0]


def _read_points_from_geojson(path):
    with open(path, 'r', encoding='utf-8') as f:
        gj = json.load(f)

    points = []

    if gj.get('type') == 'FeatureCollection':
        for feat in gj.get('features', []):
            geom = feat.get('geometry', {})
            gtype = geom.get('type')
            coords = geom.get('coordinates', [])
            if gtype == 'Point':
                points.append(coords[:2])
            elif gtype == 'MultiPoint':
                for c in coords:
                    points.append(c[:2])
    elif gj.get('type') == 'Point':
        points.append(gj.get('coordinates', [])[:2])
    elif gj.get('type') == 'MultiPoint':
        for c in gj.get('coordinates', []):
            points.append(c[:2])

    pts = np.asarray(points, dtype=float)
    if pts.ndim != 2 or pts.shape[1] != 2:
        raise RuntimeError('Could not extract 2D points from GeoJSON.')

    return pts


def _select_bc_nodes(template_data, geojson_points_file):
    default_node_bc = torch.IntTensor(np.asarray(template_data.node_BC, dtype=int))
    default_edge_bc_length = torch.FloatTensor(np.asarray(template_data.edge_BC_length, dtype=np.float32))

    if not os.path.exists(geojson_points_file):
        print(f'GeoJSON not found: {geojson_points_file}. Using template node_BC.')
        return default_node_bc, default_edge_bc_length

    inlet_pts = _read_points_from_geojson(geojson_points_file)
    if inlet_pts.shape[0] == 0:
        print('No inlet points found in GeoJSON. Using template node_BC.')
        return default_node_bc, default_edge_bc_length

    face_xy = np.asarray(template_data.mesh.face_xy)
    bc_face_ids = np.asarray(template_data.mesh.face_BC, dtype=int)
    if bc_face_ids.size == 0:
        print('Template has no face_BC entries. Using template node_BC.')
        return default_node_bc, default_edge_bc_length

    bc_face_xy = face_xy[bc_face_ids]
    distance_matrix = np.linalg.norm(inlet_pts[:, None, :] - bc_face_xy[None, :, :], axis=2)

    if inlet_pts.shape[0] <= bc_face_ids.shape[0]:
        row_ind, col_ind = linear_sum_assignment(distance_matrix)
        order = np.argsort(row_ind)
        col_ind = col_ind[order]
        selected_face_ids = bc_face_ids[col_ind]
        selected_dist = distance_matrix[row_ind[order], col_ind]
    else:
        nearest_col = np.argmin(distance_matrix, axis=1)
        selected_face_ids = bc_face_ids[nearest_col]
        selected_dist = distance_matrix[np.arange(inlet_pts.shape[0]), nearest_col]

    selected_face_ids = np.asarray(selected_face_ids, dtype=int)
    unique_selected = np.unique(selected_face_ids)

    if unique_selected.size != inlet_pts.shape[0]:
        print(
            f'Warning: {inlet_pts.shape[0]} inlet points mapped to {unique_selected.size} unique BC nodes. '
            'Check CRS and point positions.'
        )

    if len(default_edge_bc_length) == len(bc_face_ids) and inlet_pts.shape[0] <= bc_face_ids.shape[0]:
        selected_edge_lengths = default_edge_bc_length[col_ind].numpy()
    else:
        selected_edge_lengths = np.ones(len(selected_face_ids), dtype=np.float32)

    print(f'Using {len(selected_face_ids)} BC nodes from GeoJSON mapping: {selected_face_ids.tolist()}')
    print('Nearest distances:', [round(float(v), 3) for v in selected_dist])

    return torch.IntTensor(selected_face_ids), torch.FloatTensor(selected_edge_lengths.astype(np.float32))


def _get_source_points(ds):
    x = ds.coords['x'].values
    y = ds.coords['y'].values
    if x.ndim == 2 and y.ndim == 2:
        return np.column_stack([x.reshape(-1), y.reshape(-1)])
    if x.ndim == 1 and y.ndim == 1:
        xx, yy = np.meshgrid(x, y, indexing='ij')
        return np.column_stack([xx.reshape(-1), yy.reshape(-1)])
    raise RuntimeError('Unsupported x/y coordinate dimensions in SFINCS map.')


def _interpolate_time_series(source_points, grid_3d, target_points):
    if grid_3d.ndim != 3:
        raise RuntimeError('Expected 3D array [time, n, m].')

    t_steps = grid_3d.shape[0]
    n_target = target_points.shape[0]
    out = np.zeros((n_target, t_steps), dtype=np.float32)

    for t in range(t_steps):
        values = grid_3d[t].reshape(-1)
        valid = np.isfinite(values)
        if valid.sum() == 0:
            continue

        interp = griddata(
            source_points[valid],
            values[valid],
            target_points,
            method='linear',
        )

        nan_mask = ~np.isfinite(interp)
        if nan_mask.any():
            interp_nearest = griddata(
                source_points[valid],
                values[valid],
                target_points[nan_mask],
                method='nearest',
            )
            interp[nan_mask] = interp_nearest

        out[:, t] = np.nan_to_num(interp, nan=0.0).astype(np.float32)

    return out


def _build_bc_tensor(hydrograph_file, n_bc_nodes, n_time_target):
    BC = np.loadtxt(hydrograph_file)
    if BC.ndim == 1:
        BC = BC.reshape(-1, 2)
    if BC.ndim != 2 or BC.shape[1] < 2:
        raise ValueError('Hydrograph must be [time, discharge...]')

    q = BC[:, 1:]
    if q.shape[1] == 1:
        node_q = np.repeat(q.T, n_bc_nodes, axis=0)
    elif q.shape[1] == n_bc_nodes:
        node_q = q.T
    else:
        print(
            f'Warning: Hydrograph has {q.shape[1]} discharge columns, but there are {n_bc_nodes} BC nodes. '
            'Using mean discharge for all BC nodes.'
        )
        mean_q = q.mean(axis=1, keepdims=True)
        node_q = np.repeat(mean_q.T, n_bc_nodes, axis=0)

    # Resample BC to match WD time dimension if needed.
    if node_q.shape[1] != n_time_target:
        old_idx = np.linspace(0.0, 1.0, node_q.shape[1])
        new_idx = np.linspace(0.0, 1.0, n_time_target)
        node_q_resampled = np.vstack([np.interp(new_idx, old_idx, node_q[i]) for i in range(n_bc_nodes)])
    else:
        node_q_resampled = node_q

    bc_tensor = np.zeros((n_bc_nodes, n_time_target, 2), dtype=np.float32)
    bc_tensor[:, :, 1] = node_q_resampled.astype(np.float32)
    return bc_tensor


template_data = _load_template_data('./datasets/train/template_marg.pkl')
target_points = np.asarray(template_data.mesh.face_xy)

geojson_points_file = './raw_datasets_ahr/Geometry/inlet_points.geojson'
mapped_node_BC, mapped_edge_BC_length = _select_bc_nodes(template_data, geojson_points_file)

for dataset_folder, dataset_name, dataset_dir, _with_multiscale, start_sim_id, n_sim in simulation_ids:
    for sim_id in range(start_sim_id, start_sim_id + n_sim):
        netcdf_file = f'{dataset_folder}/Simulations/output_{sim_id}_map.nc'
        hydrograph_file = f'{dataset_folder}/Hydrograph/Hydrograph_{sim_id}.txt'

        if not os.path.exists(netcdf_file):
            raise FileNotFoundError(f'Missing file: {netcdf_file}')
        if not os.path.exists(hydrograph_file):
            raise FileNotFoundError(f'Missing file: {hydrograph_file}')

        ds = xr.open_dataset(netcdf_file)
        if 'zs' not in ds.data_vars or 'zb' not in ds.data_vars:
            raise RuntimeError(
                f'SFINCS variables zs/zb not found in {netcdf_file}. '
                'This notebook branch expects SFINCS map format.'
            )

        source_points = _get_source_points(ds)
        zs = ds['zs'].values
        zb = ds['zb'].values
        WD_grid = np.maximum(zs - zb[None, :, :], 0.0).astype(np.float32)

        WD = _interpolate_time_series(source_points, WD_grid, target_points)
        VX = np.zeros_like(WD, dtype=np.float32)
        VY = np.zeros_like(WD, dtype=np.float32)

        data = copy.deepcopy(template_data)
        data.WD = torch.FloatTensor(WD)
        data.VX = torch.FloatTensor(VX)
        data.VY = torch.FloatTensor(VY)
        data.node_BC = mapped_node_BC.clone()
        data.edge_BC_length = mapped_edge_BC_length.clone()

        n_bc = int(data.node_BC.shape[0])
        data.BC = torch.FloatTensor(_build_bc_tensor(hydrograph_file, n_bc, WD.shape[1]))

        out_path = os.path.join(dataset_dir, f'{dataset_name}.pkl')
        os.makedirs(dataset_dir, exist_ok=True)
        with open(out_path, 'wb') as f:
            pickle.dump([data], f)

        print(f'Saved {out_path} | WD={tuple(data.WD.shape)} | BC={tuple(data.BC.shape)} | node_BC={n_bc}')

Using 4 BC nodes from GeoJSON mapping: [485, 2232, 10996, 32731]
Nearest distances: [27683.242, 27592.902, 12810.706, 13433.181]
Saved datasets/train\ahr_river.pkl | WD=(115024, 121) | BC=(4, 121, 2) | node_BC=4
Saved datasets/test\ahr_river.pkl | WD=(115024, 121) | BC=(4, 121, 2) | node_BC=4


In [7]:
import json
import numpy as np

# Path to your GeoJSON with the 4 inlet points.
geojson_points_file = './raw_datasets_ahr/Geometry/inlet_points.geojson'

def _read_points_from_geojson(path):
    with open(path, 'r', encoding='utf-8') as f:
        gj = json.load(f)

    points = []

    if gj.get('type') == 'FeatureCollection':
        for feat in gj.get('features', []):
            geom = feat.get('geometry', {})
            gtype = geom.get('type')
            coords = geom.get('coordinates', [])
            if gtype == 'Point':
                points.append(coords[:2])
            elif gtype == 'MultiPoint':
                for c in coords:
                    points.append(c[:2])

    elif gj.get('type') == 'Point':
        points.append(gj.get('coordinates', [])[:2])
    elif gj.get('type') == 'MultiPoint':
        for c in gj.get('coordinates', []):
            points.append(c[:2])

    pts = np.asarray(points, dtype=float)
    if pts.ndim != 2 or pts.shape[1] != 2:
        raise RuntimeError('Could not extract 2D points from GeoJSON.')

    return pts

# In your current notebook workflow, BC nodes come from template_data.node_BC
# (which is inherited from template_marg.pkl).
face_xy = np.asarray(template_data.mesh.face_xy)
ghost_ids = np.asarray(template_data.node_BC, dtype=int)
bc_face_ids = np.asarray(template_data.mesh.face_BC, dtype=int)

ghost_xy = face_xy[ghost_ids]
bc_face_xy = face_xy[bc_face_ids]
inlet_pts = _read_points_from_geojson(geojson_points_file)

print(f'Loaded {len(inlet_pts)} inlet points from GeoJSON')
print(f'BC faces in template: {len(bc_face_ids)}')
print(f'BC ghost nodes in template: {len(ghost_ids)}')

# Map each inlet point to nearest BC face and nearest BC ghost node
print('\nNearest BC mapping:')
for i, p in enumerate(inlet_pts):
    d_face = np.linalg.norm(bc_face_xy - p, axis=1)
    j_face = int(np.argmin(d_face))

    d_ghost = np.linalg.norm(ghost_xy - p, axis=1)
    j_ghost = int(np.argmin(d_ghost))

    print(
        f'Point {i+1}: ({p[0]:.3f}, {p[1]:.3f}) | '
        f'nearest face_BC id={bc_face_ids[j_face]} dist={d_face[j_face]:.3f} | '
        f'nearest node_BC id={ghost_ids[j_ghost]} dist={d_ghost[j_ghost]:.3f}'
    )

print('\nTip: if distances are large, check CRS consistency between GeoJSON and mesh coordinates.')

Loaded 4 inlet points from GeoJSON
BC faces in template: 4
BC ghost nodes in template: 1

Nearest BC mapping:
Point 1: (345220.851, 5583766.788) | nearest face_BC id=2232 dist=27667.429 | nearest node_BC id=115023 dist=27668.270
Point 2: (346313.912, 5582100.558) | nearest face_BC id=2232 dist=27592.902 | nearest node_BC id=115023 dist=27595.808
Point 3: (356926.709, 5594304.264) | nearest face_BC id=32731 dist=12810.009 | nearest node_BC id=115023 dist=12804.575
Point 4: (355971.429, 5597633.049) | nearest face_BC id=32731 dist=13433.181 | nearest node_BC id=115023 dist=13427.104

Tip: if distances are large, check CRS consistency between GeoJSON and mesh coordinates.


In [11]:
# Compact status summary (post-mapping)
print('--- Dataset status ---')
print('WD shape:', tuple(data.WD.shape))
print('VX max abs:', float(np.abs(data.VX.numpy()).max()))
print('VY max abs:', float(np.abs(data.VY.numpy()).max()))
print('BC tensor shape:', tuple(data.BC.shape))
print('BC nodes:', int(data.node_BC.shape[0]))

print('\n--- Active BC nodes used in output dataset ---')
print('node_BC ids:', data.node_BC.tolist())
print('edge_BC_length size:', int(data.edge_BC_length.shape[0]))

# Verify hydrograph channels used
BC_loaded = np.loadtxt(hydrograph_file)
if BC_loaded.ndim == 1:
    BC_loaded = BC_loaded.reshape(-1, 2)
print('Hydrograph columns (incl. time):', int(BC_loaded.shape[1]))
print('Hydrograph discharge series:', int(BC_loaded.shape[1] - 1))

print('\nCheck: BC node count should match BC tensor first dimension and hydrograph series count.')
print('BC tensor first dimension:', int(data.BC.shape[0]))

--- Dataset status ---
WD shape: (115024, 121)
VX max abs: 0.0
VY max abs: 0.0
BC tensor shape: (4, 121, 2)
BC nodes: 4

--- Active BC nodes used in output dataset ---
node_BC ids: [485, 2232, 10996, 32731]
edge_BC_length size: 4
Hydrograph columns (incl. time): 5
Hydrograph discharge series: 4

Check: BC node count should match BC tensor first dimension and hydrograph series count.
BC tensor first dimension: 4
